# Week 4: LLM Benchmarking Notebook

Direct comparison: ML baseline (Ridge regression) vs LLM (Phi-3.5) on Week 3 pressure forecasting framing.
- 12-hour window -> predict next-hour pressures at 10 nodes
- Queries at least one LLM (Phi-3.5)
- Converts LLM responses into structured predictions
- Evaluates with the same regression metrics (MSE, RMSE, MAE, R2)

## Setup

In [ ]:
import os
import json
import time
import random
import re
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, Tuple, Optional
from dataclasses import dataclass

import numpy as np
import pandas as pd

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

print("Setup complete. Deferred torch/transformers imports until model load.")

## Configuration & Paths

In [ ]:
# Paths
ROOT = Path.cwd()
DATA_FILENAME = "ftes_scaled_for_GNN.csv"
DATA_CSV_CANDIDATES = [
    ROOT / DATA_FILENAME,
    ROOT / "Week 3 Deliverables" / "Data" / DATA_FILENAME,
]

DATA_CSV = None
for candidate in DATA_CSV_CANDIDATES:
    if candidate.exists():
        DATA_CSV = candidate
        break

if DATA_CSV is None:
    DATA_CSV = DATA_CSV_CANDIDATES[0]

RESULTS_ROOT = ROOT / "Results" / "week4_llm_simple"
RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_DIR = RESULTS_ROOT / RUN_TS
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Data columns
NODE_COLUMNS = [
    "tl_interval_pressure", "tl_bottom_pressure",
    "tn_interval_pressure", "tn_bottom_pressure",
    "tc_interval_pressure", "tc_bottom_pressure",
    "tu_interval_pressure", "tu_bottom_pressure",
    "ts_interval_pressure", "ts_bottom_pressure",
]
ENGINEERED_COLUMNS = ["tc_injecting", "tc_producing", "delta_P_delta_Q"]

# Window and model config
WINDOW_SIZE = 12
SUBSET_WINDOWS = 500
QUICK_MODE = True
QUICK_WINDOWS = 15  # Small; focus on correctness first

PHI_MODEL_ID = "microsoft/Phi-3.5-mini-instruct"
MAX_NEW_TOKENS = 64  # Very short; just numbers
TEMPERATURE = 0.0
RIDGE_ALPHA = 1e-3

print(f"Data CSV: {DATA_CSV}")
print(f"Results dir: {RUN_DIR}")
print(f"Evaluating: {QUICK_WINDOWS if QUICK_MODE else SUBSET_WINDOWS} windows")

## Data Loading & Windowing

In [ ]:
def build_node_features(df: pd.DataFrame) -> np.ndarray:
    """Stack 10 node pressure columns (0-axis), add engineered features (2-axis)."""
    base = np.stack([df[c].values for c in NODE_COLUMNS], axis=1)
    t = len(df)
    n = len(NODE_COLUMNS)
    
    engineered = np.zeros((t, n, 3), dtype=float)
    tc_idx = NODE_COLUMNS.index("tc_interval_pressure")
    engineered[:, tc_idx, 0] = df["tc_injecting"].values
    engineered[:, tc_idx, 1] = df["tc_producing"].values
    engineered[:, :, 2] = df["delta_P_delta_Q"].values[:, np.newaxis]
    
    return np.concatenate([base[..., np.newaxis], engineered], axis=-1)


def load_data(csv_path: Path) -> Tuple[np.ndarray, np.ndarray, pd.Series]:
    """Load CSV, build node features, create windows."""
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing: {csv_path}")
    
    df = pd.read_csv(csv_path)
    required = set(NODE_COLUMNS + ENGINEERED_COLUMNS + ["time"])
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    
    node_feat = build_node_features(df)
    
    # Create windows
    windows, targets, tstamps = [], [], []
    for i in range(len(node_feat) - WINDOW_SIZE):
        windows.append(node_feat[i:i + WINDOW_SIZE])
        targets.append(node_feat[i + WINDOW_SIZE, :, 0])  # Next-hour pressures
        tstamps.append(df["time"].iloc[i + WINDOW_SIZE])
    
    return np.stack(windows), np.stack(targets), pd.Series(tstamps)


# Load
windows, targets, timestamps = load_data(DATA_CSV)
print(f"Loaded {len(windows)} windows, shape {windows.shape}")

# Deterministic subset
n_total = len(windows)
n_subset = min(SUBSET_WINDOWS, n_total)
rng = np.random.default_rng(GLOBAL_SEED)
idx = np.arange(n_total)
rng.shuffle(idx)
subset_idx = np.sort(idx[:n_subset])

windows = windows[subset_idx]
targets = targets[subset_idx]
timestamps = timestamps.iloc[subset_idx].reset_index(drop=True)

n_eval = QUICK_WINDOWS if QUICK_MODE else len(windows)
print(f"Evaluating: {n_eval}/{len(windows)} windows (quick_mode={QUICK_MODE})")

## ML Baseline (Ridge Regression)

In [ ]:
def flatten_window(w: np.ndarray) -> np.ndarray:
    """Flatten window (12, 10, 4) → (480,)."""
    return w.reshape(-1)


def fit_ridge(X: np.ndarray, Y: np.ndarray, alpha: float) -> np.ndarray:
    """Ridge regression via normal equations."""
    xtx = X.T @ X
    reg = alpha * np.eye(X.shape[1])
    return np.linalg.solve(xtx + reg, X.T @ Y)


# Train ML on all n_eval windows
X = np.array([flatten_window(windows[i]) for i in range(n_eval)])
Y = targets[:n_eval]
w_ridge = fit_ridge(X, Y, RIDGE_ALPHA)
ml_pred = X @ w_ridge

print(f"ML trained on {n_eval} windows.")

## LLM Inference

In [ ]:
def build_simple_prompt(window: np.ndarray) -> str:
    """Build a simple text prompt asking for 10 comma-separated pressure predictions."""
    # Extract pressure history (12 timesteps, 10 nodes)
    pressure_hist = window[:, :, 0]  # (12, 10)
    
    avg_pressure = pressure_hist.mean(axis=0)
    recent_trend = pressure_hist[-1] - pressure_hist[0]
    
    lines = [
        "Based on this 12-hour pressure history at 10 nodes:",
        f"Current average pressure per node: {np.round(avg_pressure, 1).tolist()}",
        f"Pressure trend (last - first): {np.round(recent_trend, 1).tolist()}",
        "",
        "Predict the next-hour pressures at each of the 10 nodes.",
        "Return ONLY 10 comma-separated numbers (one per node).",
        "Example: 125.3, 128.5, 121.0, 119.8, 122.4, 120.1, 118.9, 117.5, 116.2, 115.0",
    ]
    return "\n".join(lines)


def extract_numbers(text: str) -> Optional[np.ndarray]:
    """Extract up to 10 numbers from text."""
    if not isinstance(text, str):
        return None
    
    # Find all numbers (including decimals and negatives)
    numbers = re.findall(r"-?\d+\.?\d*", text)
    if not numbers:
        return None
    
    try:
        floats = [float(n) for n in numbers[:10]]  # Take first 10
        if len(floats) < 10:
            return None  # Need exactly 10
        return np.array(floats, dtype=float)
    except Exception:
        return None


def load_phi_model(model_name: str):
    """Load Phi-3.5 with GPU memory detection and CPU fallback."""
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    
    torch.manual_seed(GLOBAL_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(GLOBAL_SEED)
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    
    # Detect GPU memory
    use_cuda = torch.cuda.is_available()
    if use_cuda:
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        free_gb = free_bytes / (1024 ** 3)
        print(f"Free GPU memory: {free_gb:.2f} GB")
        if free_gb < 8.0:
            print("GPU memory < 8GB; using CPU.")
            use_cuda = False
    
    dtype = torch.float16 if use_cuda else torch.float32
    model_kwargs = {"trust_remote_code": True, "dtype": dtype}
    if use_cuda:
        model_kwargs["device_map"] = "auto"
        model_kwargs["attn_implementation"] = "eager"
    
    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    if not use_cuda:
        model = model.to("cpu")
    
    model.config.use_cache = False
    if hasattr(model, "generation_config"):
        model.generation_config.use_cache = False
    model.eval()
    
    return tokenizer, model, "cuda" if use_cuda else "cpu"


def generate_llm_prediction(tokenizer, model, prompt: str) -> str:
    """Generate text from prompt."""
    import torch
    
    messages = [
        {"role": "system", "content": "You are a pressure prediction system. Return only numbers."},
        {"role": "user", "content": prompt},
    ]
    
    if hasattr(tokenizer, "apply_chat_template"):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = f"System: You are a pressure prediction system. Return only numbers.\nUser: {prompt}\nAssistant:"
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    gen_kwargs = {
        **inputs,
        "max_new_tokens": MAX_NEW_TOKENS,
        "pad_token_id": tokenizer.eos_token_id,
        "use_cache": False,
        "do_sample": False,
    }
    
    with torch.no_grad():
        out = model.generate(**gen_kwargs)
    
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


print("\nLoading Phi-3.5 model...")
tokenizer, model, device_used = load_phi_model(PHI_MODEL_ID)
print(f"Model loaded on device: {device_used}")

## Benchmark

In [ ]:
rows = []
latencies = []

for i in range(n_eval):
    y_true = targets[i]  # (10,) true pressures

    prompt = build_simple_prompt(windows[i])

    t0 = time.perf_counter()
    raw_output = generate_llm_prediction(tokenizer, model, prompt)
    dt = time.perf_counter() - t0
    latencies.append(dt)

    # Parse numbers from output
    llm_pred = extract_numbers(raw_output)
    parse_ok = llm_pred is not None and len(llm_pred) == 10

    if not parse_ok:
        llm_pred = np.full(10, np.nan, dtype=float)

    # Keep a structured parsed-prediction object for deliverable artifacts.
    parsed_prediction = {
        node: (float(llm_pred[j]) if np.isfinite(llm_pred[j]) else None)
        for j, node in enumerate(NODE_COLUMNS)
    }

    row = {
        "window_idx": i,
        "timestamp": str(timestamps.iloc[i]),
        "parse_ok": parse_ok,
        "latency_sec": dt,
        "prompt": prompt,
        "raw_output": raw_output,
        "parsed_prediction": json.dumps(parsed_prediction),
    }

    for j, node in enumerate(NODE_COLUMNS):
        row[f"true_{node}"] = float(y_true[j])
        row[f"ml_{node}"] = float(ml_pred[i, j])
        row[f"llm_{node}"] = float(llm_pred[j]) if np.isfinite(llm_pred[j]) else np.nan

    rows.append(row)

    if (i + 1) % max(1, n_eval // 5) == 0 or i + 1 == n_eval:
        print(f"  {i + 1}/{n_eval} | Parse OK: {parse_ok} | Latency: {dt:.2f}s")

results_df = pd.DataFrame(rows)
print(f"\nBenchmark complete. Parse success rate: {results_df['parse_ok'].mean():.1%}")

## Metrics & Outputs

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """MSE, RMSE, MAE, R2."""
    e = y_pred - y_true
    mse = float(np.mean(e ** 2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(e)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}


# Extract valid predictions only
valid_mask = results_df["parse_ok"].values.astype(bool)
valid_count = valid_mask.sum()

if valid_count == 0:
    print("ERROR: No valid LLM predictions parsed!")
    print("\nSample LLM outputs:")
    print(results_df[["window_idx", "parse_ok", "raw_output"]].head(10))
else:
    # Extract numpy arrays for valid rows only
    y_true_list = []
    y_ml_list = []
    y_llm_list = []
    
    for j, node in enumerate(NODE_COLUMNS):
        y_true_list.append(results_df.loc[valid_mask, f"true_{node}"].values)
        y_ml_list.append(results_df.loc[valid_mask, f"ml_{node}"].values)
        y_llm_list.append(results_df.loc[valid_mask, f"llm_{node}"].values)
    
    y_true = np.column_stack(y_true_list)
    y_ml = np.column_stack(y_ml_list)
    y_llm = np.column_stack(y_llm_list)
    
    # Overall metrics
    overall_metrics = pd.DataFrame([
        {"method": "ML_Baseline", **compute_metrics(y_true, y_ml)},
        {"method": "LLM_Phi3.5", **compute_metrics(y_true, y_llm)},
    ])
    
    print("\n=== Overall Metrics ===")
    print(overall_metrics.to_string(index=False))
    
    # Per-node metrics
    per_node_rows = []
    for j, node in enumerate(NODE_COLUMNS):
        per_node_rows.append({
            "method": "ML",
            "node": node,
            **compute_metrics(y_true[:, j], y_ml[:, j]),
        })
        per_node_rows.append({
            "method": "LLM",
            "node": node,
            **compute_metrics(y_true[:, j], y_llm[:, j]),
        })
    
    per_node_df = pd.DataFrame(per_node_rows)
    print("\n=== Per-Node Metrics (first 10 rows) ===")
    print(per_node_df.head(10).to_string(index=False))
    
    # Latency stats
    lat = np.array(latencies)
    latency_df = pd.DataFrame([{
        "count": len(lat),
        "mean_sec": lat.mean(),
        "p50_sec": np.percentile(lat, 50),
        "p95_sec": np.percentile(lat, 95),
        "max_sec": lat.max(),
    }])
    print("\n=== Latency ===")
    print(latency_df.to_string(index=False))
    
    # Save outputs
    overall_metrics.to_csv(RUN_DIR / "overall_metrics.csv", index=False)
    per_node_df.to_csv(RUN_DIR / "per_node_metrics.csv", index=False)
    latency_df.to_csv(RUN_DIR / "latency_summary.csv", index=False)
    results_df.to_csv(RUN_DIR / "row_level_results.csv", index=False)
    
    # Metadata
    meta = {
        "run_ts": RUN_TS,
        "data_csv": str(DATA_CSV),
        "window_size": WINDOW_SIZE,
        "n_eval": n_eval,
        "model": PHI_MODEL_ID,
        "device": device_used,
        "max_new_tokens": MAX_NEW_TOKENS,
        "seed": GLOBAL_SEED,
        "parse_success_rate": float(valid_count / len(results_df)),
    }
    with open(RUN_DIR / "metadata.json", "w") as f:
        json.dump(meta, f, indent=2)
    
    print(f"\n✓ Artifacts saved to {RUN_DIR}")

## Plots (optional)

In [ ]:
if valid_count > 0:
    try:
        import matplotlib.pyplot as plt
        
        # Residuals
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist((y_ml - y_true).reshape(-1), bins=40, alpha=0.7, label="ML")
        axes[0].set_title("ML Residuals")
        axes[0].set_xlabel("Residual")
        axes[0].grid(alpha=0.3)
        
        axes[1].hist((y_llm - y_true).reshape(-1), bins=40, alpha=0.7, label="LLM", color="orange")
        axes[1].set_title("LLM Residuals")
        axes[1].set_xlabel("Residual")
        axes[1].grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(RUN_DIR / "residuals.png", dpi=100, bbox_inches="tight")
        plt.close()
        
        # Pred vs true
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        lo, hi = min(y_true.min(), y_ml.min()), max(y_true.max(), y_ml.max())
        
        axes[0].scatter(y_true.reshape(-1), y_ml.reshape(-1), s=8, alpha=0.5)
        axes[0].plot([lo, hi], [lo, hi], "r--", lw=1)
        axes[0].set_xlabel("True")
        axes[0].set_ylabel("Predicted (ML)")
        axes[0].set_title(f"ML (R² = {compute_metrics(y_true, y_ml)['R2']:.3f})")
        axes[0].grid(alpha=0.3)
        
        lo, hi = min(y_true.min(), y_llm.min()), max(y_true.max(), y_llm.max())
        axes[1].scatter(y_true.reshape(-1), y_llm.reshape(-1), s=8, alpha=0.5, color="orange")
        axes[1].plot([lo, hi], [lo, hi], "r--", lw=1)
        axes[1].set_xlabel("True")
        axes[1].set_ylabel("Predicted (LLM)")
        axes[1].set_title(f"LLM (R² = {compute_metrics(y_true, y_llm)['R2']:.3f})")
        axes[1].grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(RUN_DIR / "pred_vs_true.png", dpi=100, bbox_inches="tight")
        plt.close()
        
        print("Plots saved.")
    except ImportError:
        print("matplotlib not available; skipping plots. CSV results are saved.")